# Step 5 &mdash; Training Strategy

**Goal:** train the dual-stream classifier reliably on a small, class-imbalanced, multi-label dataset.

## Recipe at a glance

| Element | Choice | Why |
|---------|--------|-----|
| Optimizer | AdamW | Standard for transformers + CNNs; decoupled weight decay |
| LR schedule | Linear warmup &rarr; cosine decay | Stable warmup, smooth annealing |
| Mixed precision | `torch.cuda.amp` | Fits Swin + dual streams into a single mid-range GPU |
| Augmentation | Synchronized brightness / contrast / hflip on **both** views | Preserve micro-defect signal |
| Sampler | Probe-level oversampling of rare classes | Boost gradient signal from rare labels |
| **Loss (Swin final)** | **Weighted `BCEWithLogitsLoss`** with `pos_weight` | Multi-label imbalance handling |
| Loss (CNN baselines) | Weighted Asymmetric Loss | More focal-loss-style for CNN exploration |

## 1. Synchronized data augmentation

The tile and global views describe the **same physical sample**. Augmenting them independently would teach the model that left-handed tiles can co-occur with right-handed global views &mdash; nonsense. Sync all transforms with the same random seed per sample.

In [ ]:
import numpy as np
import torchvision.transforms.functional as TF

class DualAugment:
    """Apply identical random transforms to both tile and global images."""

    def __init__(self, brightness=0.10, contrast=(0.90, 1.10), hflip_p=0.5):
        self.brightness = brightness
        self.contrast = contrast
        self.hflip_p = hflip_p

    def __call__(self, tile, global_img):
        bri = 1.0 + np.random.uniform(-self.brightness, self.brightness)
        con = np.random.uniform(*self.contrast)
        do_flip = np.random.rand() < self.hflip_p

        if do_flip:
            tile, global_img = TF.hflip(tile), TF.hflip(global_img)
        tile = TF.adjust_brightness(tile, bri)
        global_img = TF.adjust_brightness(global_img, bri)
        tile = TF.adjust_contrast(tile, con)
        global_img = TF.adjust_contrast(global_img, con)
        return tile, global_img

Augmentations are **conservative on purpose**: aggressive color-jitter or rotations would erase the micro-textures we are trying to classify (e.g., fine grooving).

## 2. Class imbalance: `pos_weight` for BCE

Class counts in the training set vary from 13 to 233 across the 13 labels. A naive BCE would let common classes dominate the gradient. Per-class `pos_weight = neg / pos`, clipped to avoid extreme values, balances each class's contribution:

In [ ]:
import torch
import torch.nn as nn

def compute_pos_weight(train_df, label_cols, clip_max=20.0):
    y = train_df[label_cols].values.astype(np.float32)
    pos = np.clip(y.sum(axis=0), 1.0, None)
    neg = y.shape[0] - pos
    w = np.clip(neg / pos, 1.0, clip_max)
    return torch.tensor(w, dtype=torch.float32)

# Usage:
# pos_w = compute_pos_weight(train_df, label_cols).to(device)
# criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

## 3. Probe-level oversampling of rare classes

On top of `pos_weight`, we also **resample** rare-class probes more often. Critically the boost is applied at the **probe level** (not the tile level) so the model still sees the full set of tiles for a probe together.

In [ ]:
from torch.utils.data import WeightedRandomSampler

def build_sampler_probe_level(train_df, rare_classes, upsample_boost=3.0):
    rare = [c for c in rare_classes if c in train_df.columns]
    if not rare:
        w = np.ones(len(train_df), dtype=np.float32)
    else:
        has_rare = train_df[rare].eq(1).any(axis=1)
        rare_probe_ids = train_df.loc[has_rare, "probe_id"].unique()
        in_rare_probe = train_df["probe_id"].isin(rare_probe_ids).values
        w = np.ones(len(train_df), dtype=np.float32)
        w[in_rare_probe] = float(upsample_boost)
    return WeightedRandomSampler(w, num_samples=len(w), replacement=True)

## 4. Warmup + cosine LR schedule

Linear warmup avoids destabilizing the pretrained backbone in the first few steps; cosine decay anneals smoothly to a small floor by the end of training.

In [ ]:
import math

def build_warmup_cosine_scheduler(optimizer, total_steps, warmup_steps, min_lr_ratio=0.0):
    def lr_lambda(step):
        s = max(0, min(step, total_steps))
        if warmup_steps > 0 and s < warmup_steps:
            return s / max(1, warmup_steps)
        if total_steps == warmup_steps:
            return 1.0
        progress = (s - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

## 5. Core training loop (AMP + grad clipping)

The training loop itself is intentionally minimal &mdash; the heavy lifting lives in the model, loss, sampler, and scheduler defined above.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler,
                    device, grad_clip=1.0, amp=True):
    model.train()
    running = 0.0
    for tile, global_img, y in loader:
        tile = tile.to(device, non_blocking=True)
        global_img = global_img.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp):
            logits = model(tile, global_img)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        if grad_clip > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()
    return running / max(1, len(loader))

## 6. Checkpoint selection

We monitor **macro-F1** (or **mAP**) on validation each epoch and keep the best checkpoint. Macro-F1 is preferred because it weights all 13 classes equally, which matches our deployment goal of "every defect type matters".